# Week 12 — Kaggle Capstone
**FissionLab · AI/ML Foundations · Aarush**

**W12 (Aug 17):** Your first real ML project from start to finish.

**Choose one:**
- **Titanic Survival** (classification) — kaggle.com/competitions/titanic
- **House Prices** (regression) — kaggle.com/competitions/house-prices-advanced-regression-techniques

This notebook is your scaffold. Replace the placeholder data with your Kaggle download.
The goal is a clear README, honest results, and a notebook saved to your Google Drive.

**Reference:** HOML 3rd ed., Ch. 2 (end-to-end pipeline)

In [ ]:
# Core imports — all available in Colab by default
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, r2_score
)

RANDOM_STATE = 42
print('Imports ready.')

---
## Step 1 — Load Data
Upload your Kaggle CSV files to Colab or mount Google Drive.
For Titanic: `train.csv` and `test.csv`.

In [ ]:
# Option A: upload manually
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv('train.csv')

# Option B: mount Drive (recommended — files stay between sessions)
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/fissionlab/kaggle/titanic/train.csv')

# --- PLACEHOLDER (delete when you have real data) ---
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame.rename(columns={'target': 'Survived'})
print('Placeholder dataset loaded. Replace with your Kaggle data.')
print(df.shape)
df.head()

---
## Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Overview ===')
print('Shape:', df.shape)
print('\nNull counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nDescriptive stats:')
df.describe().round(2)

In [ ]:
# Distribution of target variable
plt.figure(figsize=(6, 3))
df['Survived'].value_counts().plot(kind='bar', color=['#ff6b6b', 'steelblue'])
plt.title('Target Distribution')
plt.xlabel('Class'); plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

print('Class balance:', df['Survived'].value_counts(normalize=True).round(3).to_dict())

### Your EDA (TODO)
Add cells here to explore:
- Distribution of each feature (histograms)
- Correlation heatmap
- Feature vs target plots
- Any interesting patterns you find

In [ ]:
# TODO: correlation heatmap
# plt.figure(figsize=(12, 8))
# sns.heatmap(df.corr(), annot=False, cmap='RdBu_r')
# plt.title('Feature Correlations')
# plt.show()

---
## Step 3 — Preprocessing

In [ ]:
# Using the placeholder data
feature_cols = [c for c in df.columns if c != 'Survived']
X = df[feature_cols].values
y = df['Survived'].values

# Impute missing values (median for numerics)
imputer = SimpleImputer(strategy='median')
X = imputer.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

---
## Step 4 — Train Multiple Models and Compare

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=10000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}

results = {}
for name, model in models.items():
    # Cross-validation on training set (5-fold)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = {'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(), 'test': test_acc}
    print(f'{name:25s}: CV={cv_scores.mean():.3f}±{cv_scores.std():.3f}, Test={test_acc:.3f}')

best_name = max(results, key=lambda k: results[k]['test'])
print(f'\nBest model: {best_name} (test accuracy = {results[best_name]["test"]:.3f})')

---
## Step 5 — Evaluate Best Model

In [ ]:
best_model = models[best_name]
y_pred = best_model.predict(X_test)

print(f'=== {best_name} — Final Evaluation ===')
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix — {best_name}')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.show()

---
## Step 6 — Reflection and Write-Up
Replace the prompts below with your actual findings.

### Project Summary

**Dataset:** [Name your dataset and source]

**Problem type:** [Classification / Regression]

**Best model:** [Model name and key hyperparameters]

**Final test accuracy (or RMSE):** [Your result]

### What I Tried
1. 
2. 
3. 

### What Worked / What Didn't
*Be honest — imperfect results are fine. Explain why you think something didn't work.*

### What I Would Try Next
*One idea you'd try with more time.*

### Connection to HOML
*Which chapter(s) were most useful for this project?*

---
## Save to Google Drive
Run this cell to save a copy of this notebook to your shared FissionLab Drive folder.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# src = '/content/12_capstone.ipynb'
# dst = '/content/drive/MyDrive/fissionlab/capstone_submission.ipynb'
# shutil.copy(src, dst)
# print(f'Saved to {dst}')